# PostgreSQL versus Oracle AI Database for agent memory

Compare memory writes and retrieval using Oracle AI Database 26ai and PostgreSQL with pgvector and pg_textsearch. This notebook uses 100 synthetic memories and eight queries. It demonstrates transaction visibility, Oracle Text synchronization, SQL property graphs, retrieval evaluation, and database row policies.

Both engines receive identical 384-dimensional vectors generated by Oracle's local ONNX model. Oracle also supplies the same BGE cross-encoder scores for both pipelines. PostgreSQL's inference is external in this experiment; no PostgreSQL in-engine inference claim follows from these results.

This is a functional walkthrough with a small, editorially judged relevance fixture. It does not measure production relevance, ANN recall, throughput, or latency. Managed-provider availability is a documentation finding in the article. The optional native Deep Data Security section tests local end-user authentication with row and cell grants on Oracle Free; it does not test external identity propagation.

### Reading this notebook

Run the cells in order, or read the saved outputs without starting a database. The walkthrough covers setup and seed data, transaction and graph behavior, retrieval quality at depths eight and ten, then security and cleanup. The ten-candidate table supplies the article's figures.

Implementation details appear in expandable **View Python** blocks with syntax highlighting. Ranking examples show one selected question at a time; edit the selector variables to inspect another question, engine, or method. The judgment and ranking inspection cells can run independently after a kernel restart. Ranking cells load the included reference result when no experiment is in memory and label the source explicitly. Definitions and reading instructions appear immediately above each result table.


## Setup

Use a Python 3.11+ kernel. The first code cell installs the pinned runtime packages. Keep this folder's supporting files beside the notebook, and add DB_USER, DB_PASSWORD, DB_DSN, and PG_DSN to the Developer Hub repository-root .env. The helper searches upward from this folder; an existing folder-local .env or explicit PG_ORACLE_ENV_FILE also works. Exported environment variables take precedence.

From the repository root, open `jupyter lab notebooks/postgresql_vs_oracle_agent_memory/postgresql_vs_oracle_agent_memory.ipynb`. Start both databases before running the cells in order. Restart the kernel before a full rerun. See the [README](README.md) for PostgreSQL container startup, required privileges, and the optional DDS configuration.

An existing ALL_MINILM_L12_V2 model is reused. If absent, the notebook downloads Oracle's augmented MiniLM model and loads it through a BLOB; EMBEDDING_ONNX_FILE can supply a local artifact instead. BGE_RERANKER must already be loaded or supplied as an augmented ONNX artifact through RERANKER_ONNX_FILE. Its inputs must remain FIRST_INPUT and SECOND_INPUT. The README links the export walkthrough.

Each run uses `PG_ORACLE_MEMORY` in Oracle and `postgres_oracle.memory` in PostgreSQL. Setup recreates the notebook's fixed objects, clearing leftovers from an interrupted run. Use one active notebook run per PostgreSQL database; a session lock rejects overlapping runs. The final cell removes the notebook objects and preserves existing models. PostgreSQL setup requires extension and role-administration privileges; RLS checks switch to a non-owner role without bypass privileges.


In [7]:
# Restart the kernel if pip replaces a package already imported in this session.
%pip install -q --disable-pip-version-check oracledb==26.1.0 python-dotenv==1.2.4 "psycopg[binary]==3.3.6"

Note: you may need to restart the kernel to use updated packages.


In [8]:
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path
from IPython.display import Markdown, display
import oracledb
oracledb.defaults.program = "devrel-developerhub-postgresql-versus-oracle-ai-database-for-agent-memory"
if not (Path.cwd() / "lab.py").is_file():
    raise RuntimeError(
        "Open this notebook from the repository root with "
        "jupyter lab notebooks/postgresql_vs_oracle_agent_memory/postgresql_vs_oracle_agent_memory.ipynb; "
        "keep the supporting files in the notebook folder."
    )
from corpus import DOCS, QUERIES, validate_fixture
from lab import MemoryLab
import notebook_display as view

if "lab" in globals():
    lab.cleanup()
lab = MemoryLab()
view.table("Connected databases", ["Oracle", "PostgreSQL", "Python"],
           [(lab.report["oracle"], lab.report["postgres"], lab.report["python"])])
program = lab.oq("SELECT SYS_CONTEXT('USERENV', 'CLIENT_PROGRAM_NAME') FROM dual")[0][0]
display(Markdown(f"Connection program: `{program}`"))


Oracle,PostgreSQL,Python
23.26.3.0.0,18.6 (Debian 18.6-1.pgdg12+2),3.12.5


Connection program: `devrel-developerhub-postgresql-versus-oracle-ai-database-for-agent-memory`

## Inference inside Oracle

`VECTOR_EMBEDDING` runs the augmented MiniLM model in Oracle. The BGE cross-encoder runs through `DBMS_VECTOR_CHAIN.UTL_TO_RERANK` with `provider=database`. The returned scores are logits, not probabilities. The model readiness check also uses `PREDICTION` to verify the model's input signature.

In [9]:
lab.models()
view.table("Local model readiness", ["Embedding dimensions", "Reranker test logit"],
           [(lab.report["checks"]["in_database_models"]["dimensions"],
             f'{lab.report["checks"]["in_database_models"]["rerank_logit"]:.4f}')],
           "The embedding dimension is checked against 384. The finite reranker logit confirms inference works; it is not a probability or a relevance grade.")
view.source(MemoryLab.database_rerank, "Calls the local ONNX reranker through DBMS_VECTOR_CHAIN.")


Embedding dimensions,Reranker test logit
384,-3.1473


## Synthetic memories and relevance judgments

The fixture contains 100 memories: 79 active Acme records, nine superseded Acme records, and 12 records belonging to another tenant. Only active Acme memories are eligible for retrieval. The corpus includes direct answers, supporting evidence, and distractors about similar projects, identifiers, people, and policies.

[qrels.json](qrels.json) scores all 800 question-memory pairs and gives each a rationale. These grades are editorial judgments fixed for the run. They are not predictions from the embedding or reranking models, independent human judgments, or a held-out test set.

| Grade | Meaning in this experiment | Gain used by nDCG |
|---|---|---:|
| 3 | Direct answer, or an explicit actionable part of the answer | 7 |
| 2 | Substantive support or a useful partial answer | 3 |
| 1 | Useful qualification or execution prerequisite | 1 |
| 0 | Irrelevant or excluded by tenant/lifecycle policy | 0 |

A shared name or keyword alone is insufficient for a positive grade. Superseded and other-tenant memories always receive zero. Fixture validation checks complete coverage, valid grades, rationales, and eligibility. The result file records source and qrels hashes.

The **0-3 scale is a common convention, with task-specific definitions**. [NIST's TREC Deep Learning guidance](https://trec.nist.gov/data/deep2021.html) also uses four levels, but its passage grade 1 means merely related and is excluded from binary relevance. Here, grade 1 means useful context and counts toward recall. Our rubric and threshold must accompany the reported scores.

The next cell lists the questions and shows positive judgments for one selected question. Change `judgment_query` to inspect another; all zero and positive judgments remain available in `qrels.json`.

Lexical queries use a fixed keyword selection shared by both engines. Oracle receives OR-separated words; pg_textsearch receives the same words in its native query format. Tokenization and scoring still differ.


In [10]:
import notebook_display as view
from corpus import validate_fixture

judgment_query = "q1"  # Choose q1 through q8.
validate_fixture()
view.fixture(judgment_query)


Memories,Eligible,Questions,Judged pairs,Positive pairs
100,79,8,800,39


ID,Question,Relevant memories
q1,Who must approve the Cedar production release?,5
q2,How do I fix checkout error AX417?,4
q3,When must customer conversations and their summaries be removed?,5
q4,How should I send Mina a routine project update?,4
q5,Can a support agent refund 750 dollars without additional approval?,5
q6,What should a client do after a rate limit response?,4
q7,Which region should handle live production API traffic?,8
q8,Why did Atlas stop ingesting and how do I recover it?,4


Memory,Grade,Memory text,Reason
m01,3 · Direct answer,"For Project Cedar, Priya is the release approver. Production rollout requires her written approval.",Names Priya and requires her written approval for production rollout.
m02,2 · Support,"Project Cedar deployment checklist: run migrations, verify health checks, then request release approval.","Describes the Cedar release-approval step, but does not identify the approver."
m27,3 · Direct answer,Cedar approval history: Marco approved the initial prototype. Priya now owns production approval.,Explicitly names Priya as the current production approver; Marco is historical.
m29,3 · Direct answer,Cedar production release authorization: attach Priya's written sign-off to the rollout ticket before deploying.,Requires Priya's written sign-off on the production rollout ticket.
m30,3 · Direct answer,"If Cedar release approval is missing, contact Priya and hold the rollout. Passing automated checks does not replace her authorization.",Directs the agent to contact Priya and hold the rollout until she authorizes it.


## Tables and indexes

The memory tables are `PG_ORACLE_MEMORY` in Oracle and `postgres_oracle.memory` in PostgreSQL. Each memory has content, tenant, lifecycle status, and a vector. Edges refer to memory rows. Oracle's property graph is defined over the relational tables. Oracle Text uses explicit manual synchronization so the freshness boundary is observable. PostgreSQL uses a BM25 index with English text configuration.

The corpus is too small to assess approximate indexes, so vector queries perform exact cosine searches. The implementation is in [lab.py](lab.py); the following cell displays and executes its schema definitions.

In [11]:
view.source(MemoryLab.setup, "Creates the memory tables, graph objects, and search indexes.")
lab.setup()
view.table("Search extensions", ["PostgreSQL extension", "Version"], list(lab.report["extensions"].items()))

view.table("Memory tables", ["Engine", "Table"],
           [("Oracle", lab.mem), ("PostgreSQL", lab.schema + ".memory")])


PostgreSQL extension,Version
vector,0.8.7
pg_textsearch,1.5.0


Engine,Table
Oracle,PG_ORACLE_MEMORY
PostgreSQL,postgres_oracle.memory


## Write the same memories to both engines

Oracle computes the vector in the INSERT statement. PostgreSQL receives that vector as a bound value. Copying the vector isolates storage and retrieval from embedding-model differences; it is not a recommended cross-database deployment.

In [12]:
view.source(MemoryLab.insert_oracle, "Computes the memory embedding in the INSERT statement.")
lab.seed()
view.table("Identical seed data", ["Memories in each database", "Check"],
           [(lab.report["checks"]["identical_seed_counts"]["documents"], "PASS")])


Memories in each database,Check
100,PASS


## Rollback, commit, and search visibility in Oracle

Insert a memory with its ONNX embedding and a graph edge. An independent connection sees neither row before commit. Rollback removes both; a subsequent commit publishes both. A graph query in the writing transaction can already see its edge.

The committed probe is initially absent from the manually synchronized text index. After `CTX_DDL.SYNC_INDEX`, it becomes searchable. This observation applies to this configured CONTEXT index, not every Oracle index mode.

In [13]:
lab.oracle_transactions()
view.transactions(lab, "oracle")


Observation,Memory rows,Edge rows
Writing connection before commit,1,1
Independent connection before commit,0,0
After rollback,0,0
After commit,1,1


Before explicit sync,After explicit sync
0,1


Check,Result
oracle transaction and text freshness,PASS


## PostgreSQL atomic writes and BM25 visibility

The same test inserts a precomputed vector and a relational edge in one transaction. Another connection sees neither before commit, and rollback removes both. The committed probe is immediately searchable through pg_textsearch in this configuration. Atomic memory writes do not require inference to execute in the database process.

In [14]:
lab.pg_transactions()
view.transactions(lab, "postgres")


Observation,Memory rows,Edge rows
Writing connection before commit,1,1
Independent connection before commit,0,0
After rollback,0,0
After commit,1,1


Matches after commit
1


Check,Result
postgres transaction and bm25 freshness,PASS


## Traverse provenance with SQL property graphs

The edge links the release-approval memory to its deployment checklist. `GRAPH_TABLE` returns relational rows that can be joined to retrieval results. PostgreSQL can query these same relationships with joins or recursive CTEs; this notebook does not install Apache AGE.

In [15]:
view.source(MemoryLab.graph_rows, "Traverses the provenance edge using GRAPH_TABLE.")
rows = lab.graph_rows()
lab.check("sql_property_graph", rows == [("m01", "m02")], edges=rows)
view.table("Provenance edge returned by the graph query", ["Source memory", "Related memory"], rows,
           "m01 is the Cedar approval policy; m02 is its deployment checklist.")
view.check_summary(lab, ["sql_property_graph"])


Source memory,Related memory
m01,m02


Check,Result
sql property graph,PASS


## Candidate retrieval and fusion

Each branch selects at most eight active Acme memories, with a stable ID tie-break. Reciprocal rank fusion adds `1 / (60 + rank)` across branches and retains eight candidates. Reranking then sorts those candidates with the BGE cross-encoder.

Both implementations fuse in SQL. Oracle uses `CONTAINS`/`SCORE`; PostgreSQL uses pg_textsearch BM25. This code does not use `DBMS_HYBRID_VECTOR.SEARCH` or a hybrid vector index. Tenant and lifecycle predicates here are explicit query filters. The later security cell tests database enforcement separately.

In [16]:
view.source(MemoryLab.retrieve_oracle, "Oracle vector and lexical branches, followed by SQL RRF.")
view.source(MemoryLab.retrieve_pg, "PostgreSQL vector and BM25 branches, followed by SQL RRF.")


## Evaluate every query

**nDCG@5 measures the quality of the first five results and their ordering.** It rewards high-grade memories near the top. A score of 1 means the best possible top five under the qrels; 0 means none of those results is relevant.

This notebook uses exponential gain and a base-2 logarithmic rank discount:

$$
DCG@k = \sum_{r=1}^{k} \frac{2^{grade_r}-1}{\log_2(r+1)},
\qquad nDCG@k = \frac{DCG@k}{IDCG@k}
$$

`r` starts at 1. `IDCG@k` is the ideal DCG obtained by sorting **all judged memories** by descending grade and taking the best `k`. The first result receives its full gain; the second receives about 63%; the third receives 50%. This makes placing a direct answer first more valuable than placing it third.

The gain function `2**grade - 1` is an established variant, used by [PyTorch Ignite](https://docs.pytorch.org/ignite/master/generated/ignite.metrics.rec_sys.NDCG.html). Linear gain is also valid: [scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ndcg_score.html) uses the supplied relevance values directly. Passing raw 0-3 grades there will not reproduce this notebook's exponential-gain scores.

**Recall@8 measures coverage in the first eight results.** It divides the number of retrieved memories with grade >= 1 by the total number of memories with grade >= 1 for that question. Finding three of four relevant memories gives `3 / 4 = 0.75`. Recall gives grades 1, 2, and 3 equal weight and ignores ordering within the first eight.

A list can have high recall but lower nDCG if it finds the useful memories and puts background context ahead of direct answers. Empty result lists score zero. Each metric is calculated per question, then averaged equally across the eight questions. The later ten-candidate evaluation uses the same definitions at `k=10`.

| Method in the output | Retrieval path |
|---|---|
| Dense vectors | Exact cosine similarity |
| Lexical search | Oracle Text or PostgreSQL pg_textsearch BM25 |
| Vector + lexical (RRF) | SQL fusion of the two ranked branches |
| RRF + BGE reranking | The same fused candidates reordered by BGE |

The BGE score for a query-memory pair is computed through `DBMS_VECTOR_CHAIN.UTL_TO_RERANK` with `provider=database` and shared by both pipelines. PostgreSQL's reranked results include external inference. Runtime is not compared.


In [17]:
# Keep per-query progress messages out of the report; execution errors still surface.
with redirect_stdout(StringIO()):
    summary = lab.evaluate()
view.scores(summary, depth=8)


Engine,Method,Mean nDCG@5,Mean Recall@8
Oracle,Dense vectors,0.8993,0.9062
Oracle,Lexical search,0.8332,0.8875
Oracle,Vector + lexical (RRF),0.9151,0.9375
Oracle,RRF + BGE reranking,0.9542,0.9375
PostgreSQL,Dense vectors,0.8993,0.9062
PostgreSQL,Lexical search,0.8991,0.9844
PostgreSQL,Vector + lexical (RRF),0.9346,0.9531
PostgreSQL,RRF + BGE reranking,0.9542,0.9531


## Inspect one question's ranking

The averages above can hide a missed answer. This section starts with **one question**, compares its scores across methods, and then displays the complete eight-result list for your selected engine and method.

Read the detailed table in rank order. The **Grade** column is the expected usefulness from qrels. The **nDCG@5 contribution** column shows how much each of the first five positions adds to the normalized score. Positions six through eight contribute to recall but are outside the nDCG@5 cutoff. A grade-0 memory near the top takes a position that could have held useful evidence.

The final table lists relevant memories that the candidate list missed. Those omissions reduce recall. Change `query_id`, `engine`, or `method` below to investigate a different result; the selected view reads the recorded rankings and does not rerun retrieval.

This inspection cell also works after a kernel restart. It imports its display helper and uses the current kernel's experiment results when available; otherwise it loads the included `results/verification.json`. The output identifies the source and run timestamp. No database connection or model execution is needed.


In [18]:
import notebook_display as view

query_id = "q5"       # q1 through q8; q6 asks how to handle a rate-limit response.
engine = "oracle"     # "oracle" or "postgres"
method = "hybrid"     # "vector", "lexical", "hybrid", or "hybrid_rerank"
rows = view.retrieval_results(8, report=getattr(globals().get("lab"), "report", None))
view.ranking(rows, query_id=query_id, engine=engine, method=method)


**Results source:** current kernel experiment. Run started: 2026-11-06T22:34:42.190773+00:00. This cell reads recorded rankings; it does not connect to either database.

Engine,Method,nDCG@5,Recall@8
Oracle,Dense vectors,0.9284,1.0000
Oracle,Lexical search,0.5850,0.6000
Oracle,Vector + lexical (RRF),0.7959,1.0000
Oracle,RRF + BGE reranking,0.9697,1.0000
PostgreSQL,Dense vectors,0.9284,1.0000
PostgreSQL,Lexical search,0.9688,1.0000
PostgreSQL,Vector + lexical (RRF),0.9600,1.0000
PostgreSQL,RRF + BGE reranking,0.9697,1.0000


Rank,Memory,Grade,nDCG@5 contribution,Memory text
1,m53,3 · Direct answer,0.4218,A 750-dollar refund exceeds the support agent's 500-dollar authority. Obtain a finance manager's approval before executing the refund.
2,m15,3 · Direct answer,0.2661,Refunds above 500 dollars require approval from a finance manager before the refund tool can execute.
3,m17,1 · Qualification,0.0301,The refund tool accepts a purchase identifier and an amount in cents. It must receive an idempotency key.
4,m16,2 · Support,0.0779,Refunds up to 500 dollars may be issued by the support agent after the purchase is verified.
5,m27,0 · Irrelevant,0.0000,Cedar approval history: Marco approved the initial prototype. Priya now owns production approval.
6,m01,0 · Irrelevant,Outside top 5,"For Project Cedar, Priya is the release approver. Production rollout requires her written approval."
7,m02,0 · Irrelevant,Outside top 5,"Project Cedar deployment checklist: run migrations, verify health checks, then request release approval."
8,m54,3 · Direct answer,Outside top 5,"When a requested refund exceeds 500 dollars, the support workflow pauses at the finance-approval step. A verified purchase alone is insufficient."


**Reading this result:** 5 of 5 relevant memories appear in the list (Recall@8 = 1.000). A grade-3 answer below grade-0 or grade-1 text is an ordering problem that nDCG penalizes. Coverage details appear below.

**No relevant memories are missing from this candidate list.**

## Evaluate ten candidates

This run sets both branch depth and the fused candidate pool to ten. Corpus, qrels, models, filters, and the RRF constant stay fixed. The table reports nDCG@5 alongside nDCG@10 and Recall@10 so you can compare the first five positions and the full candidate list.

**These are the ten-candidate results reported in the article.** A wider pool can recover useful memories missed at depth eight; reranking can then change their order. A wider pool does not guarantee a better score.

The helper also runs the inline embedding and Oracle Text probes shown separately below.


In [19]:
from review_experiments import run_review_experiments
followups = run_review_experiments(lab)
view.scores(followups["retrieval_depth10"]["summary"], depth=10)


Engine,Method,Mean nDCG@5,Mean nDCG@10,Mean Recall@10
Oracle,Dense vectors,0.8993,0.9397,0.9375
Oracle,Lexical search,0.8332,0.9024,0.9437
Oracle,Vector + lexical (RRF),0.9392,0.9739,0.9844
Oracle,RRF + BGE reranking,0.9600,0.9817,0.9844
PostgreSQL,Dense vectors,0.8993,0.9397,0.9375
PostgreSQL,Lexical search,0.8991,0.9665,0.9844
PostgreSQL,Vector + lexical (RRF),0.9346,0.9742,0.9844
PostgreSQL,RRF + BGE reranking,0.9632,0.9825,0.9844


## Compare RRF order with BGE order

Reranking changes the order of the fused candidate list. It cannot retrieve a missing memory, so **recall stays the same**. The summary table shows each question's nDCG@10 before and after reranking. Positive changes mean the ordering improved under the qrels; negative changes mean it became worse.

The selected question then gets a rank-movement table. Each row is one memory, with its relevance grade, position before reranking, and position afterward. Rows are sorted in BGE order. Look for direct answers moving above background context, and check whether irrelevant records move upward.

The default is `q7`, the production-region question. Its missing-memory table makes the difference between candidate coverage and ranking quality visible. You can select either engine or another question below. The grades are editorial judgments; a broader relevance study needs independent judgments and a held-out set.

This inspection cell also works after a kernel restart. It imports its display helper and uses the current kernel's experiment results when available; otherwise it loads the included `results/verification.json`. The output identifies the source and run timestamp. No database connection or model execution is needed.


In [20]:
import notebook_display as view

query_id = "q2"    # q1 through q8; q7 asks which region handles production API traffic.
engine = "oracle"  # "oracle" or "postgres"
rows = view.retrieval_results(10, report=getattr(globals().get("lab"), "report", None))
view.reranking(rows, query_id=query_id, engine=engine)


**Results source:** current kernel experiment. Run started: 2026-11-06T22:34:42.190773+00:00. This cell reads recorded rankings; it does not connect to either database.

Query,Engine,RRF nDCG@10,BGE nDCG@10,Change,Recall@10
q1,Oracle,0.9712,1.0000,+0.0288,1.000
q1,PostgreSQL,0.8901,1.0000,+0.1099,1.000
q2,Oracle,0.9619,0.9919,+0.0299,1.000
q2,PostgreSQL,0.9563,0.9919,+0.0356,1.000
q3,Oracle,0.9537,0.9908,+0.0371,1.000
q3,PostgreSQL,0.9952,0.9908,-0.0044,1.000
q4,Oracle,0.9829,0.9709,-0.0120,1.000
q4,PostgreSQL,1.0000,0.9709,-0.0291,1.000
q5,Oracle,0.9641,0.9697,+0.0055,1.000
q5,PostgreSQL,0.9814,0.9697,-0.0118,1.000


Memory,Grade,RRF rank,BGE rank,Movement,Memory text
m35,3 · Direct answer,1,1,Same,"AX417 recovery procedure: flush the checkout resolver's DNS cache, confirm the new endpoint address, then retry the failed checkout."
m05,3 · Direct answer,2,2,Same,Incident AX417: a stale DNS cache caused checkout failures. Flush the resolver cache and retry the checkout.
m36,3 · Direct answer,4,3,Up 1,The AX417 investigation found that checkout workers cached the old DNS answer after an endpoint change. Resolver cache invalidation restored connectivity.
m40,0 · Irrelevant,8,4,Up 4,The incident dashboard can export AX417 and AX471 charts as PNG files. Chart colors follow the reporting theme.
m07,2 · Support,5,5,Same,Checkout recovery guide: resolver cache invalidation fixes stale DNS answers after an endpoint change.
m37,0 · Irrelevant,3,6,Down 3,AX418 checkout failures affect browsers with an expired cart cookie. Clear the cart cookie and rebuild the basket.
m39,0 · Irrelevant,6,7,Down 1,The checkout address form validates postal codes against the selected country before submitting an order.
m06,0 · Irrelevant,9,8,Up 1,Incident AX471: payment authorization timed out because the upstream payment provider was unavailable.
m54,0 · Irrelevant,10,9,Up 1,"When a requested refund exceeds 500 dollars, the support workflow pauses at the finance-approval step. A verified purchase alone is insufficient."
m38,0 · Irrelevant,7,10,Down 3,AX471 payment failures require waiting for the upstream authorization provider to recover. Flushing DNS does not resolve that incident.


**Reading this result:** nDCG@10 changes from 0.9619 to 0.9919 (+0.0299). Recall@10 stays at 1.000. Relevance grades come from qrels.json; they are not BGE model scores.

**No relevant memories are missing from this candidate list.**

## Query embedding and Oracle Text behavior

These probes test database behavior on separate synthetic data. They do not contribute to the relevance scores above.

The first SQL statement embeds raw question text inside the vector search. The synchronization table then compares when another connection can search newly written text under MANUAL and ON COMMIT index settings. Finally, the proximity query compares nearby terms with a simple AND match. Read the document counts and matching IDs to see the behavior each check verifies.


In [21]:
view.text_probes(lab, followups)


### Query embedding inside the executed SQL

The raw-text query returned the same ordered memory IDs as the separately embedded query. The check verifies equivalent results for this question.

Mode,Before commit,After commit,After explicit sync,Rolled-back probe
MANUAL,0,0,3,0
ON COMMIT,0,3,3,0


### The executed proximity query

Query,Matching fixture IDs
connection AND pooling,"adjacent, far, nearby"
"NEAR((connection, pooling), 5)","adjacent, nearby"


## Database-enforced row policies

A fixed `tenant_id = 'acme'` policy is applied to each experiment memory table. Oracle uses VPD under a non-SYS owner without `EXEMPT ACCESS POLICY`. PostgreSQL uses RLS and a separate non-owner role without `BYPASSRLS` or superuser privileges. Queries that omit tenant filters still return only Acme rows, including lexical queries.

This verifies row-policy enforcement. It does not authenticate end users, establish agent identity, test cell masking, or exercise Deep Data Security. A production application needs a trusted identity path and policies on every protected relationship.

In [22]:
lab.scope_and_security()
c = lab.report["checks"]["database_row_policies"]
view.table("Database-enforced tenant filtering", ["Engine", "Other-tenant rows visible"],
           [("Oracle", c["oracle_other_rows"]), ("PostgreSQL", c["postgres_other_rows"])],
           "Both policies are fixed to Acme. This checks row filtering, not end-user authentication.")
view.check_summary(lab, ["database_row_policies"])


Engine,Other-tenant rows visible
Oracle,0
PostgreSQL,0


Check,Result
database row policies,PASS


## Native Deep Data Security on Oracle Free

This optional example uses CREATE END USER, CREATE DATA ROLE, and CREATE DATA GRANT. It authenticates two local end users and runs the same SELECT without an application tenant predicate. Both users can read the Acme rows. The support user can read the payment amount only on the support-owned case; the other amount is masked as NULL. Finance can read both amounts. The Globex row is inaccessible to both. An OR 1=1 predicate does not expand their access, and an incorrect password fails authentication.

For a local development container, explicitly set PG_ORACLE_DDS_CONTAINER and PG_ORACLE_DDS_PDB as described in the README. Administrative DDL runs through local SQL*Plus OS authentication. The demo creates temporary end users, roles, and data grants, and removes them in a finally block. It does not grant privileges to the existing notebook account. With no container configured, it reports not_run. For another development deployment, supply a DBA executor to run_dds.

The saved demonstration ran on Oracle AI Database Free 23.26.3.0.0. It tests native row and cell controls with local password authentication; external IAM tokens, pooled connections, and agent delegation are separate integration work. The VPD test above remains a distinct experiment. See Oracle's [local end-user setup](https://docs.oracle.com/en/database/oracle/oracle-database/26/ddscg/configure-database-local-end-user-authentication.html) and [data grants](https://docs.oracle.com/en/database/oracle/oracle-database/26/ddscg/fine-grained-data-authorization.html).


In [23]:
from dds_experiment import configured_dds
dds = configured_dds(lab)
view.dds_results(lab, dds)


**DDS was not run.** Set PG_ORACLE_DDS_CONTAINER and PG_ORACLE_DDS_PDB for the optional local Free demonstration, or call run_dds with your DBA executor.

## Save reproducible results

The run-specific JSON in `results/run-<timestamp>.json` records database and extension versions, model metadata, source hashes, per-query rankings, and checks. These files are ignored by Git; the recorded results/verification.json remains a reference snapshot. Reused model artifacts are identified by database name and creation date. A newly loaded artifact also gets a SHA-256 hash.


In [24]:
saved = lab.save()
view.table("Saved reference for this run", ["Result file", "Passed checks", "Failed checks"],
           [(saved["result"], saved["passed"], saved["failed"])])


Result file,Passed checks,Failed checks
run-20261106T223442190740Z.json,117,0


## Remove this run's objects

Run this cell even after an earlier cell fails. It removes this run's tables, property graph, policy function, PostgreSQL schema, and test role. Existing models and other articles' objects remain available.

In [25]:
cleanup = lab.cleanup()
view.table("Cleanup", ["Run objects removed", "Existing models preserved"],
           [(cleanup["run_objects_removed"], cleanup["existing_models_preserved"])])


Run objects removed,Existing models preserved
True,True
